# Course 8: Mathematics and AI
## Session 8: Sheaves on graphs

**Goal:** build a sheaf Laplacian, see why it does not oversmooth the way the graph Laplacian does, and compare sheaf diffusion with a GCN on a heterophilic graph.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every Python cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a library, a GPU or a Lean installation. Lean code in this course was written carefully but was NOT compiled here: check it in Lean before you teach it.

### 1. A heterophilic graph
In a heterophilic graph, neighbours tend to have different labels (think of a fraud network where fraudsters link to ordinary users). Averaging over neighbours, which is what a GCN does, then mixes the classes.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
rng = np.random.default_rng(0); torch.manual_seed(0)
n = 300; y = np.repeat([0, 1], n // 2); sgn = 2.0 * y - 1                    # labels as -1 / +1
P_edge = np.where(y[:, None] == y[None, :], 0.03, 0.03)                      # as many edges inside a class as between classes
A = np.triu(rng.random((n, n)) < P_edge, 1); A = (A | A.T).astype(np.float32)
X = (sgn[:, None] * 0.35 * np.ones((1, 16)) + rng.normal(size=(n, 16))).astype(np.float32)      # weak signal in the features
deg = np.maximum(A.sum(1), 1)
same = (A * (y[:, None] == y[None, :])).sum() / A.sum()
print(f"nodes {n} | edges {int(A.sum() / 2)} | share of edges joining the same class: {same:.2f}")

### 2. Graph diffusion forgets the classes
Repeated neighbour averaging converges to a constant on a connected graph: the null space of the graph Laplacian is spanned by the all-ones vector. That is oversmoothing.

In [ ]:
L = np.eye(n) - A / np.sqrt(np.outer(deg, deg))
x = sgn + rng.normal(size=n)                                                 # one noisy channel that starts out informative
corr = lambda v: abs(np.corrcoef(v, sgn)[0, 1])
h = x.copy(); track = [corr(h)]
for _ in range(30): h = h - 0.5 * L @ h; track.append(corr(h))
print("correlation with the labels after 0, 5, 30 steps of graph diffusion:", [round(track[i], 3) for i in (0, 5, 30)])

### 3. A sheaf: a small vector space on every node and a map on every edge end
A cellular sheaf puts a vector space (a 'stalk') on each node and each edge, and a linear restriction map from each node to each edge it touches. Two neighbours agree when their restricted values match on the shared edge. The sheaf Laplacian measures disagreement:

`(L_F x)_i = sum over neighbours j of  F_i^T (F_i x_i - F_j x_j)`

With 1-dimensional stalks and maps of +1 or -1, 'agree' can mean 'be equal' on some edges and 'be opposite' on others.

In [ ]:
def sheaf_laplacian(A, S):
    """1-dimensional stalks. S[i, j] = F_i * F_j is +1 if the edge asks its ends to agree, -1 if it asks them to be opposite."""
    d = np.maximum(A.sum(1), 1)
    return np.eye(len(A)) - (S * A) / np.sqrt(np.outer(d, d))

S_true = np.outer(sgn, sgn)                                                  # the ideal sheaf: -1 exactly on edges that join different classes
L_F = sheaf_laplacian(A, S_true)
print("the label vector is in the null space of the ideal sheaf Laplacian:", np.allclose(L_F @ (np.sqrt(deg) * sgn), 0, atol=1e-5))
h = x.copy(); track_s = [corr(h)]
for _ in range(30): h = h - 0.5 * L_F @ h; track_s.append(corr(h))
print("correlation with the labels after 0, 5, 30 steps of sheaf diffusion:", [round(track_s[i], 3) for i in (0, 5, 30)])
assert track_s[30] > 0.95 and track[30] < track[0]

Same graph, different geometry. Graph diffusion drives everything to a constant and the class signal fades. Sheaf diffusion with the right maps converges to the labels themselves. This is the argument of Bodnar et al. (2022): oversmoothing and heterophily are both statements about which sheaf you diffuse on.

### 4. Learn the sheaf from data
In practice nobody hands you the ideal maps. Neural sheaf diffusion learns them from the node features: for each edge, a small network looks at both ends and outputs the edge's sign. Everything is trained end to end from the node labels.

In [ ]:
A_t = torch.tensor(A); Xt = torch.tensor(X); yt = torch.tensor(y); deg_t = torch.tensor(deg, dtype=torch.float32)
idx = rng.permutation(n); train, test = idx[:60], idx[60:]
norm_A = A_t / torch.sqrt(deg_t[:, None] * deg_t[None, :])
ei = A_t.nonzero()

class SheafNet(nn.Module):
    def __init__(self, d=16, hid=32, steps=4):
        super().__init__()
        self.inp = nn.Linear(d, hid); self.edge = nn.Sequential(nn.Linear(2 * hid, hid), nn.ReLU(), nn.Linear(hid, 1)); self.out = nn.Linear(hid, 2); self.steps = steps
    def signs(self, H):
        i, j = ei[:, 0], ei[:, 1]
        s = torch.tanh(self.edge(torch.cat([H[i], H[j]], 1)) + self.edge(torch.cat([H[j], H[i]], 1))).squeeze(1)    # symmetric in the two ends
        S = torch.zeros_like(A_t); S[i, j] = s; return S
    def forward(self, X):
        H = torch.tanh(self.inp(X)); S = self.signs(H)
        for _ in range(self.steps): H = H - 0.5 * (H - (S * norm_A) @ H)        # one step of sheaf diffusion
        return self.out(H), S

class GCNNet(nn.Module):
    def __init__(self, d=16, hid=32, steps=4):
        super().__init__(); self.inp = nn.Linear(d, hid); self.out = nn.Linear(hid, 2); self.steps = steps
    def forward(self, X):
        H = torch.tanh(self.inp(X))
        for _ in range(self.steps): H = H - 0.5 * (H - norm_A @ H)              # the same diffusion with all signs +1
        return self.out(H), None

class MLPNet(nn.Module):
    def __init__(self, d=16, hid=32): super().__init__(); self.net = nn.Sequential(nn.Linear(d, hid), nn.Tanh(), nn.Linear(hid, 2))
    def forward(self, X): return self.net(X), None

def fit(make, seeds=5):
    accs = []
    for sd in range(seeds):
        torch.manual_seed(sd); m = make(); opt = torch.optim.Adam(m.parameters(), lr=0.01, weight_decay=1e-3)
        for _ in range(200):
            out, _ = m(Xt); loss = F.cross_entropy(out[train], yt[train]); opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad(): out, S = m(Xt)
        accs.append((out[test].argmax(1) == yt[test]).float().mean().item())
    return np.mean(accs), np.std(accs), m, S

r_mlp, r_gcn, r_sheaf = fit(MLPNet), fit(GCNNet), fit(SheafNet)
for name, r in [("MLP (no graph)", r_mlp), ("graph diffusion (GCN-like)", r_gcn), ("learned sheaf diffusion", r_sheaf)]:
    print(f"{name:28s} test accuracy {r[0]:.3f} ± {r[1]:.3f}")

### 5. Did it learn the right sheaf?
Compare the learned edge signs with the ideal ones (+1 inside a class, -1 across).

In [ ]:
S = r_sheaf[3].numpy(); mask = A > 0
agree = (np.sign(S[mask]) == S_true[mask]).mean()
print(f"learned edge signs that match the ideal sheaf: {agree:.2f}  (0.50 would be chance)")
print("mean learned sign on same-class edges: %+.2f | on cross-class edges: %+.2f" % (S[mask & (S_true > 0)].mean(), S[mask & (S_true < 0)].mean()))

Report these numbers as they come out. On this graph the plain diffusion model has nothing to gain from the edges, because half of them point the wrong way. Whether the learned sheaf beats the MLP depends on how well the edge network can guess the signs from noisy features, and that is the honest test of the method. The full method of Bodnar et al. uses stalks of dimension greater than 1 and orthogonal or general restriction maps; this notebook uses the simplest case.

### TODO
1. Change the graph to be strongly homophilic (0.05 inside, 0.005 across). Which model wins now?
2. Give the sheaf model the ideal signs `S_true` instead of learned ones. That is the ceiling. How far below it is the learned model?
3. Stalks of dimension 2: replace each sign with a 2x2 rotation matrix. Write the block sheaf Laplacian and check that it is symmetric and positive semi-definite.